In [305]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [306]:
df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


### The code categorizes 'quality' into binary (0 for bad, 1 for good), prints the occurrences of each category
#### *If the **quality value >= 6**, it means the quality is **good** and I define it as **1**.*
#### *If the **quality value < 6,** it means the quality is **bad** and I define it as **0**.*

In [307]:
df['quality'] = df['quality'].apply(lambda x: 0 if x in [3, 4, 5] else 1)

quality_counts = df['quality'].value_counts()
print(quality_counts)

quality
1    787
0    671
Name: count, dtype: int64


In [308]:
X = df.drop(columns=['quality'])
y = df['quality']
standard_scaler = StandardScaler()
X = standard_scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [309]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

In [310]:
def calculate_bias_variance(model, X_test, y_test):
    predictions = model.predict(X_test)

    # Calculating bias
    mean_prediction = np.mean(predictions)
    true_mean = np.mean(y_test)
    bias = mean_prediction - true_mean

    # Calculating variance
    variance = np.mean((predictions - mean_prediction) ** 2)

    print(f"Bias: {bias}")
    print(f"Variance: {variance}")

    bias_threshold = 0.15
    variance_threshold = 0.2

    if abs(bias) > bias_threshold and variance < variance_threshold:
        print("Model Status: Underfitting - High bias and low variance indicate the model is too simplistic.")
    elif abs(bias) < bias_threshold and variance > variance_threshold:
        print("Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.")
    elif abs(bias) > bias_threshold and variance > variance_threshold:
        print(
            "Model Status: Unbalanced - Both bias and variance are high, suggesting the model may not generalize well.")
    else:
        print("Model Status: Balanced - The model has a good balance between bias and variance.")

 ### Here I train and evaluate SVM models with linear and RBF kernels using all features. It uses cross-validation to calculate training and testing accuracy, and also measures bias and variance to assess model performance.

In [311]:
# Linear
svc_linear = SVC(kernel='linear')
svc_linear.fit(X_train, y_train)

cv_results = cross_validate(svc_linear, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'SVC LINEAR:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(svc_linear, X_test, y_test)

SVC LINEAR:
Training Accuracy: 75.017%
Testing Accuracy: 72.36%
Bias: -0.047945205479452024
Variance: 0.2490500093826234
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


In [312]:
# RBF
svc_rbf = SVC(kernel='rbf', class_weight='balanced')
svc_rbf.fit(X_train, y_train)

cv_results = cross_validate(svc_rbf, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'SVC RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(svc_rbf, X_test, y_test)

SVC RBF:
Training Accuracy: 80.555%
Testing Accuracy: 73.115%
Bias: -0.07191780821917804
Variance: 0.24995308688309248
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### Both models show overfitting, indicated by low bias, high variance, and higher training than testing accuracy—implying they fit training data well but do not generalize effectively. The linear kernel may offer better interpretability, while the RBF kernel might capture more complex patterns.

### I will now build the two models again using the features selected by the KBest algorithm, which applies the f_classif scoring function. This approach will help determine if feature selection improves model performance.

In [313]:
'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

### The code builds and evaluates SVM models using linear and RBF kernels. Each model is trained with selected features, and cross-validation is used to determine training and testing accuracies, as well as bias and variance.

In [314]:
# Linear
svc_linear = SVC(kernel='linear')
svc_linear.fit(X_train, y_train)

cv_results = cross_validate(svc_linear, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'SVC LINEAR:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(svc_linear, X_test, y_test)

SVC LINEAR:
Training Accuracy: 75.017%
Testing Accuracy: 72.36%
Bias: -0.044520547945205435
Variance: 0.2488271720773128
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


In [315]:
# RBF
svc_rbf = SVC(kernel='rbf', class_weight='balanced')
svc_rbf.fit(X_train, y_train)
cv_results = cross_validate(svc_rbf, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'SVC RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(svc_rbf, X_test, y_test)

SVC RBF:
Training Accuracy: 80.555%
Testing Accuracy: 73.115%
Bias: -0.07876712328767121
Variance: 0.25
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### Both SVM models (linear and RBF) are overfitting, as indicated by low bias and relatively high variance. Despite feature selection, the RBF kernel achieves a higher testing accuracy but still struggles with generalization.


## Hyperparam tuning
### The code performs hyperparameter tuning on SVM models (RBF and linear) using GridSearchCV to find the best parameters for optimal accuracy.

In [318]:
#RBF
param_grid_rbf = {
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization parameter
    'gamma': [0.001, 0.01, 0.1, 1, 10],  # Kernel coefficient for RBF
    'class_weight': [None, 'balanced']  # Balancing class weights to handle imbalance
}


svc_rbf = SVC(kernel='rbf')

grid_search = GridSearchCV(estimator=svc_rbf, param_grid=param_grid_rbf, cv=5, scoring='accuracy', verbose=1)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

# Output training and testing accuracy
print(f'SVC RBF with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)  
calculate_bias_variance(best_model, X_test, y_test)  


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'C': 1, 'class_weight': None, 'gamma': 1}
Best Cross-Validation Accuracy: 0.7564
SVC RBF with Best Parameters:
Training Accuracy: 84.088%
Testing Accuracy: 72.635%
Bias: 0.0
Variance: 0.24379574028898476
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


In [317]:
#Linear
param_grid_linear = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # Regularization parameter
    'class_weight': [None, 'balanced']     # Class weighting to handle imbalanced data
}

svc_linear = SVC(kernel='linear')

grid_search = GridSearchCV(estimator=svc_linear, param_grid=param_grid_linear, cv=5, scoring='accuracy', verbose=1)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

cv_results = cross_validate(best_model, X, y, cv=5, return_train_score=True)  
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

# Output training and testing accuracy
print(f'SVC LINEAR with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)  
calculate_bias_variance(best_model, X_test, y_test)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters: {'C': 0.01, 'class_weight': None}
Best Cross-Validation Accuracy: 0.7333
SVC LINEAR with Best Parameters:
Training Accuracy: 74.211%
Testing Accuracy: 73.457%
Bias: -0.07191780821917804
Variance: 0.24995308688309248
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### Despite tuning, both models still show overfitting. The tuned RBF kernel had a testing accuracy of 72.635%, which was slightly lower compared to the untuned model's 73.115%, indicating overfitting persists. For the linear kernel, testing accuracy improved from 72.36% before tuning to 73.457% after tuning. However, both models still struggle to generalize well, showing persistent overfitting.